<a href="https://colab.research.google.com/github/nikolay909694/StatPrac/blob/main/EDa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Прогнозирование индекса Мосбиржи по новостям Telegram‑канала  
## Разведочный анализ данных и создание финального датасета

### Постановка задачи

Мы хотим предсказать **направление движения индекса Мосбиржи** (растёт / падает)  
на следующий торговый день, опираясь исключительно на **тексты новостей** из публичного Telegram‑канала **banksta**.

Такая постановка типична для **сентимент-анализа в финансах**:  
тексты могут отражать настроения участников рынка, ожидания и инсайды,  
которые влияют на цены в краткосрочной перспективе.

### Используемые метрики

В задачах бинарной классификации (рост vs падение) обычно применяют:

- **Accuracy** – доля правильных ответов. Проста, но может вводить в заблуждение при сильном дисбалансе классов.

- **ROC‑AUC** – площадь под ROC‑кривой, показывает способность модели разделять классы независимо от порога. Хороша как общая метрика.


Так как данные представляют собой временной ряд, для честной оценки применяют **кросс‑валидацию с учётом времени** (TimeSeriesSplit), чтобы не заглядывать в будущее.

### План работы

1. Загрузить и проанализировать историю цен, создать бинарные метки «рост / падение» на следующий день.
2. Загрузить агрегированные дневные новости и изучить их статистику.
3. Преобразовать тексты в векторные представления (эмбеддинги) с помощью мультиязычной модели Sentence‑Transformer.
4. Объединить метки и эмбеддинги в один набор данных с общим временны́м индексом.
5. Сохранить результат в файлы `labels_final.csv` и `embeddings.npy`, готовые к подаче в модель.
6. Провести визуальную проверку качества эмбеддингов с помощью PCA.

In [ ]:
!pip install -q sentence-transformers pandas numpy matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")

## 1. Цены → метки

In [ ]:
px = pd.read_csv('/content/sample_data/imoex.csv')
px.columns = [c.strip().lower() for c in px.columns]
px['date'] = pd.to_datetime(px['date']).dt.date

px = px.sort_values('date').reset_index(drop=True)
px['close_next'] = px['close'].shift(-1)
px = px.dropna(subset=['close_next']).copy()
px['label_close_next'] = (px['close_next'] > px['close']).astype(int)

# Оставляем только нужное
labels_df = px[['date', 'label_close_next']].copy()

print(f"Дней с метками: {labels_df.shape[0]}")
display(labels_df.head())

In [ ]:

px['return'] = px['close_next'] / px['close'] - 1.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(px['return'], bins=50, kde=True, color='steelblue', ax=axes[0])
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Дневные доходности')

sns.countplot(x='label_close_next', data=px, palette=['#e74c3c', '#2ecc71'], ax=axes[1])
axes[1].set_title('Классы: 0 – падение, 1 – рост')
axes[1].set_xticklabels(['Падение', 'Рост'])
plt.tight_layout()
plt.show()
print(f"Доля роста: {px['label_close_next'].mean():.2%}")

## 2. Новости → эмбеддинги

In [ ]:
news = pd.read_csv('/content/sample_data/daily_news.csv')
news['date'] = pd.to_datetime(news['date']).dt.date
print(f"Дней с новостями: {news.shape[0]}")
display(news.head())

In [ ]:

news['text_len'] = news['text_concat'].fillna('').apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(news['n_msgs'], bins=30, color='steelblue', ax=axes[0])
axes[0].set_title('Число постов в день')
sns.histplot(news['text_len'], bins=50, color='teal', ax=axes[1])
axes[1].set_title('Длина текста за день')
plt.tight_layout()
plt.show()

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
texts = news['text_concat'].fillna('').tolist()

print(f"Кодируем {len(texts)} текстов...")
embeddings = model.encode(texts, batch_size=16, show_progress_bar=True,
                          normalize_embeddings=True)
print(f"Форма эмбеддингов: {embeddings.shape}")

## 3. Объединение и сохранение

Теперь у нас есть:
- `labels_df`: дата и метка
- `news`: дата и эмбеддинг (пока отдельно)

Приводим их к общему набору дат, сохраняем метки и эмбеддинги **в одном порядке**.

In [ ]:
# Оставляем только даты, присутствующие в обоих наборах
common_dates = set(labels_df['date']) & set(news['date'])
print(f"Общих дней: {len(common_dates)}")

labels_final = labels_df[labels_df['date'].isin(common_dates)].sort_values('date').reset_index(drop=True)
news_final = news[news['date'].isin(common_dates)].sort_values('date').reset_index(drop=True)


assert list(labels_final['date']) == list(news_final['date']), "Даты не совпадают!"


final_embeddings = embeddings[news_final.index]


labels_final.to_csv('labels_final.csv', index=False)
np.save('embeddings.npy', final_embeddings)

print("   Файлы созданы:")
print("   labels_final.csv  — дата и label_close_next")
print("   embeddings.npy    — массив эмбеддингов (N x 384)")

## 4. Проверка

Убедимся, что файлы корректны и соответствуют друг другу.

In [ ]:
# Загрузим обратно и проверим
l = pd.read_csv('labels_final.csv')
e = np.load('embeddings.npy')

print(f"Меток: {l.shape[0]}")
print(f"Эмбеддингов: {e.shape[0]}")
print(f"Размерность вектора: {e.shape[1]}")
print("\nПервые 5 дат из labels_final.csv:")
display(l.head())
print("\nПервые 5 значений первого измерения эмбеддинга:")
print(e[:5, 0])

## 5. Визуальное представление эмбеддингов



In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
emb_2d = pca.fit_transform(e)

plt.figure(figsize=(8, 6))
for lbl, color, name in [(0, '#e74c3c', 'Падение'), (1, '#2ecc71', 'Рост')]:
    mask = l['label_close_next'] == lbl
    plt.scatter(emb_2d[mask, 0], emb_2d[mask, 1], c=color, alpha=0.6, s=10, label=name)
plt.title('PCA проекция финальных эмбеддингов')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Выводы

 **Что получено**:
- Метки лежат в `labels_final.csv` (date, label_close_next)
- Эмбеддинги лежат в `embeddings.npy` (матрица N×384)
- Порядок строк одинаковый — i-я строка labels соответствует i-й строке embeddings

